# Perda de acesso a POIs — Porto Alegre

Cada cenário condicionado pode afetar a acessibilidade de duas formas distintas,
e é preciso separá-las:

- **Perda de acesso** — o POI deixa de ser alcançável. Acontece quando uma
  penalização interdita arestas (`custo = inf`), cortando o grafo.
- **Perda de tempo** — o POI continua alcançável, mas o trajeto fica mais longo.

Em Porto Alegre, só o **movimento de massa** interdita arestas. A **inundação**
apenas encarece o deslocamento. Por isso os dois grupos de cenários produzem
mapas de naturezas diferentes, e este notebook trata cada um separadamente.

A referência é sempre o cenário **típico**: um par é considerado perdido quando
é alcançável no típico e deixa de ser no condicionado.


## 1. Bibliotecas e caminhos


In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colors as mpl_colors
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch

BASE_DIR = Path.cwd().parent.parent.parent
RESULTS_DIR = BASE_DIR / "results" / "porto_alegre"
MATRIZES = RESULTS_DIR / "matrices"
OUTPUT_DIR = Path.cwd()

CENARIOS = ["inundacao", "movimento_massa", "combinado"]
ROTULOS = {
    "typical": "Típico",
    "inundacao": "Inundação",
    "movimento_massa": "Movimento de massa",
    "combinado": "Combinado",
}

print("matrizes:", MATRIZES)
print("existe:", MATRIZES.exists())


matrizes: /home/matheus/Documentos/mestrado/dissertacao/results/porto_alegre/matrices
existe: True


## 2. Cálculo da perda por célula

Para cada célula, a perda é a **fração dos seus pares que deixaram de ser
alcançáveis**:

$$\text{perda}_i = \frac{\#\{j : t^{\text{típico}}_{ij} < \infty \ \wedge\ t^{\text{cond}}_{ij} = \infty\}}{\#\{j : t^{\text{típico}}_{ij} < \infty\}}$$

O denominador usa só os pares alcançáveis no típico, então uma célula que já
estava isolada não entra como se tivesse piorado.


In [2]:
def carregar_matriz(cenario):
    """Índice (cell_idx, poi_idx) -> travel_time."""
    m = pd.read_parquet(MATRIZES / f"matrix_{cenario}.parquet")
    return m.set_index(["cell_idx", "poi_idx"])["travel_time"]


tipico = carregar_matriz("typical")
alcancavel_tipico = np.isfinite(tipico)
print(f"típico: {len(tipico)} pares, {alcancavel_tipico.sum()} alcançáveis")

perda = {}
for cenario in CENARIOS:
    cond = carregar_matriz(cenario)
    cond = cond.reindex(tipico.index)
    alcancavel_cond = np.isfinite(cond)
    perdido = alcancavel_tipico & ~alcancavel_cond

    por_celula = perdido.groupby("cell_idx").mean()
    denominador = alcancavel_tipico.groupby("cell_idx").sum()
    # normaliza pela fração dos pares que existiam no típico
    por_celula = perdido.groupby("cell_idx").sum() / denominador
    perda[cenario] = por_celula

    n_afetadas = int((por_celula > 0).sum())
    print(f"\n{ROTULOS[cenario]:20s} pares perdidos {int(perdido.sum()):6d} "
          f"({100 * perdido.mean():5.1f}%)")
    print(f"{'':20s} células afetadas {n_afetadas:5d} / {len(por_celula)} "
          f"({100 * n_afetadas / len(por_celula):.1f}%)")
    print(f"{'':20s} perda por célula: mediana {por_celula.median():.3f}  "
          f"p90 {por_celula.quantile(.9):.3f}  "
          f"células com perda total {int((por_celula >= 1).sum())}")


típico: 52296 pares, 52296 alcançáveis

Inundação            pares perdidos      0 (  0.0%)
                     células afetadas     0 / 4358 (0.0%)
                     perda por célula: mediana 0.000  p90 0.000  células com perda total 0

Movimento de massa   pares perdidos  14577 ( 27.9%)
                     células afetadas  2066 / 4358 (47.4%)
                     perda por célula: mediana 0.000  p90 1.000  células com perda total 954

Combinado            pares perdidos  14577 ( 27.9%)
                     células afetadas  2066 / 4358 (47.4%)
                     perda por célula: mediana 0.000  p90 1.000  células com perda total 954


## 3. Mapa da perda de acesso

A escala vai de 0 a 1 e é a mesma nos três mapas. A leitura é direta: quanto mais
escuro, maior a fração de POIs que a célula **deixou de alcançar**.

O cinza marca as células fora da área de estudo — as que não têm nenhum par na
matriz, por estarem fora do município ou não tocarem a rede viária.


In [3]:
grid = gpd.read_parquet(RESULTS_DIR / "prepared" / "grid.parquet")

# Perda total = 1 em 954 células. Escala contínua de 0 a 1, com destaque
# para o extremo: quem perdeu tudo fica na cor mais escura.
CMAP = "magma_r"
NORM = mpl_colors.Normalize(0, 1, clip=True)

malhas = {}
for cenario in CENARIOS:
    malha = grid[["cell_idx", "geometry"]].merge(
        perda[cenario].rename("perda").reset_index(), on="cell_idx", how="left"
    )
    malhas[cenario] = gpd.GeoDataFrame(malha, geometry="geometry", crs=grid.crs)

    n = int(malha["perda"].notna().sum())
    print(f"{ROTULOS[cenario]:20s} células no mapa: {n:5d} / {len(grid)}")


Inundação            células no mapa:  4358 / 5065
Movimento de massa   células no mapa:  4358 / 5065
Combinado            células no mapa:  4358 / 5065


In [ ]:
LARGURA = 7.0
ALTURA = LARGURA * (37.6 / 27.7)

fig, eixos = plt.subplots(
    1, 3, figsize=(3 * LARGURA + 0.6, ALTURA + 1.8),
    gridspec_kw={"wspace": 0.02},
)

for ax, cenario in zip(eixos, CENARIOS):
    malhas[cenario].plot(
        column="perda", cmap=CMAP, norm=NORM, ax=ax,
        edgecolor="none", missing_kwds={"color": "#d9d9d9"},
    )
    ax.set_aspect("equal")
    ax.set_anchor("C")
    ax.set_axis_off()
    ax.set_title(ROTULOS[cenario], fontsize=17, fontweight="bold", pad=12)

cax = fig.add_axes([0.30, 0.075, 0.40, 0.017])
barra = fig.colorbar(
    ScalarMappable(norm=NORM, cmap=CMAP), cax=cax,
    orientation="horizontal", ticks=np.arange(0, 1.01, 0.2),
)
barra.set_label("Fração de POIs que deixaram de ser alcançáveis",
                fontsize=14, labelpad=12)
barra.ax.tick_params(labelsize=12)

fig.suptitle("Perda de acesso a POIs por cenário — Porto Alegre",
             fontsize=20, y=0.97)
destino = OUTPUT_DIR / "porto_alegre_perda_acesso.png"
fig.savefig(destino, dpi=240, bbox_inches="tight")
plt.close(fig)
print("salvo:", destino.name)
